---
<a id='week5'></a>
# Week 5 — Scenario Analysis *(Optional — Complete Only if Time Permits)*

> ⚠️ **Proceed only if Weeks 3 and 4 are fully complete, documented, and committed to GitHub.** If time is limited, skip to Week 6.

**Learning Objective:** Build a what-if scenario module to simulate emissions policy impact.


### Setup — Reload Data

This notebook runs independently of Weeks 3–4: it needs only the filtered actuals from Week 1 (`data/ghg_filtered.csv`). It reloads them for two jobs — the grey historical reference line (§5.3) and the training data for the **BAU baseline, which §5.2 fits itself**: a refit of Week 4's ETS(A,Ad,N) on *every* observed year, rather than a reuse of Week 4's exported forecast (§5.2 explains why).

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tools.sm_exceptions import ValueWarning
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from constants import *

EXPANDED_COUNTRIES = get_expanded_countries()

df_filtered = pd.read_csv('../data/ghg_filtered.csv')

<a id='51-design'></a>
### 5.1 Scenario Design

| Scenario | Label | Applied from 2025 |
|----------|-------|-------------------|
| A | Business as Usual (BAU) | ETS(A,Ad,N) forecast, fitted on all observed years (1990–2024), unchanged |
| B | Moderate Mitigation | −2% per year applied to BAU |
| C | Aggressive Mitigation | −5% per year applied to BAU |

**✏️ Basis and limitations:**

**Business as Usual (BAU)** is *not* "emissions stay flat" — it is the ETS(A,Ad,N) damped-trend forecast, **refitted on every observed year (1990–2024)** so that the pathway continues from where the data actually ends (§5.2 explains why this is a second fit rather than Week 4's export). That forecast already embeds each country's own historical trajectory, including any structural decline already underway (e.g. the UK's post-Climate-Change-Act decline, Germany's plateau). So "no policy change" here specifically means "no *additional* intervention beyond what's already baked into the fitted trend" — a scenario where China's trend keeps decelerating and the UK keeps declining is still BAU, because those are properties of the historical data, not the scenario logic.

**Limits of the BAU baseline** (the scenarios inherit all of them): it is ETS(A,Ad,N) only, with no model selection or comparison with other model families; it covers the expanded set only, and an aggregate mixes large and small emitters, so a total can look better than the typical country; the 2020 COVID dip is left in the training data (no outlier handling — §5.2 gives the evidence); single-year data anomalies are flagged but not altered (§5.2b: Kuwait 1991, the oil fires, distorts Kuwait's fit); it is a point forecast with no uncertainty band; and it extrapolates territorial fossil-fuel and cement CO₂ only. The evidence in §5.2 shows this fit is more accurate than Week 4's over the next 2–3 years and shows no visible COVID distortion — **it does not establish forecast skill over the 16-year scenario horizon.**

**Moderate Mitigation (−2%/year)** and **Aggressive Mitigation (−5%/year)** are flat, uniform compounding reduction rates applied identically to all countries in the expanded set (`EXPANDED_COUNTRIES`, ~40). They are illustrative bounding cases, not derived from any underlying causal model — there is no link to GDP growth, energy-mix transition, carbon pricing, or technology deployment cost curves. Applying one percentage uniformly ignores enormous real-world heterogeneity: a −2%/year path is a modest ask for a country already decarbonizing its grid (e.g. Germany), but would require an unprecedented policy shift for a coal-dependent, still-industrializing economy (e.g. South Africa or India). The −5%/year Aggressive case is roughly the order of magnitude of the UK's actual historical decline rate discussed in Week 4's §4.4 trend interpretation — useful as a real-world sanity check on scale, but sustaining it for 16 consecutive years across every expanded-set country simultaneously assumes uninterrupted policy commitment, no economic shocks or rebounds (the 2020 COVID dip-and-rebound visible throughout Weeks 3–4 is a reminder that real trajectories are rarely this smooth), and technology deployment at a pace historically achieved by only a handful of economies for short windows.

**Overall limitation:** these three scenarios are best read as *comparative, bounding cases* — a floor (BAU), a middle path, and an aggressive-but-plausible-scale path — rather than calibrated forecasts of what any specific country's policy trajectory will actually look like.

<a id='52-calc'></a>
### 5.2 Scenario Calculation

Compute annual projections from 2025 to 2040 for each country and scenario.

**Why BAU is a second ETS fit, not Week 4's export.** Week 4 fits its ETS on data through 2018 (`TRAIN_CUTOFF`) so that 2019–2023 can be *held out* for the forecast-vs-actual chart. That is the right choice for **evaluation**, but it is the wrong input for a **scenario baseline**: a forecast that stops learning in 2018 has never seen 2019–2024, so a BAU pathway built from it starts in 2025 at a level that does not continue from the observed 2024 data. Reusing it here starts the scenarios **+3.7% above the observed 2024 total** (per-country steps ranging from −51% to +62%), so every scenario carries a discontinuity that has nothing to do with mitigation. The fix is two fits, not a changed cutoff: Week 4's fit stays as it is (its teaching job is the forecast-vs-actual demonstration), and this section fits the **same model on every observed year** for use as the scenario baseline.

**What the evidence says (and does not).** Fitting through 2021 or 2022 — *with* the COVID years inside the training data — and forecasting the following years gave a median per-country error of 3.5% / 6.4% / 6.8% (2022–24) and 3.9% / 5.5% (2023–24), against 6.7% / 9.4% / 11.1% and 9.4% / 11.1% for the fit stuck at 2018; the expanded-set total was within −1.6% to −0.4% instead of +2.2% to +2.4%. Interpolating over the 2020 dip gave no consistent benefit, so none is applied. This is a short-horizon check on two cutoffs: it supports using the full-data fit as the baseline, but it is **not** evidence of skill over a 16-year horizon.

```python
# Outline — implement the details
# fit ExponentialSmoothing(trend='add', damped_trend=True, seasonal=None) per country on 1990–2024
# bau_2025_2040 = fit.forecast(16)
# years_elapsed = np.arange(len(bau_2025_2040))  # 0, 1, 2, ...
# moderate   = bau_2025_2040 * (1 - 0.02) ** years_elapsed
# aggressive = bau_2025_2040 * (1 - 0.05) ** years_elapsed
```

Store in a tidy long-format DataFrame: `country, year, scenario, co2_projected`.
Save as `data/scenario_projections.csv`.

In [ ]:
# BAU baseline: ETS(A,Ad,N) refitted on EVERY observed year, then the three scenarios for every expanded-set country
SCENARIO_START, SCENARIO_END = 2025, 2040
LAST_OBSERVED = int(df_filtered['year'].max())
assert SCENARIO_START == LAST_OBSERVED + 1, (
    f"the data now ends in {LAST_OBSERVED}: the scenarios must start in {LAST_OBSERVED + 1} so BAU continues from the last observation"
)

bau_forecasts = {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ValueWarning)
    warnings.simplefilter('ignore', FutureWarning)   # statsmodels: "no supported index" for a year-indexed series (same filter as Week 4 §4.3)
    for country in EXPANDED_COUNTRIES:
        series = (
            df_filtered[df_filtered['country'] == country]
            .sort_values('year').set_index('year')['co2']
        )
        assert series.index.tolist() == list(range(series.index.min(), LAST_OBSERVED + 1)) and series.notna().all(), \
            f"{country}: the training series must be gap-free through {LAST_OBSERVED}"
        fit = ExponentialSmoothing(series, trend='add', damped_trend=True, seasonal=None).fit(optimized=True)
        forecast = fit.forecast(SCENARIO_END - LAST_OBSERVED)
        forecast.index = np.arange(LAST_OBSERVED + 1, SCENARIO_END + 1)
        bau_forecasts[country] = forecast

def bau_series(country, start, end):
    """BAU baseline (ETS fit on all observed years) for `country`, sliced to years [start, end]."""
    return bau_forecasts[country].loc[start:end]

REDUCTION_RATES = {'BAU': 0.0, 'Moderate': 0.02, 'Aggressive': 0.05}
SCENARIO_YEARS = np.arange(SCENARIO_START, SCENARIO_END + 1)

scenario_rows = []
for country in EXPANDED_COUNTRIES:
    bau_2025_2040 = bau_series(country, SCENARIO_START, SCENARIO_END)
    years_elapsed = np.arange(len(bau_2025_2040))
    for scenario, rate in REDUCTION_RATES.items():
        projected = bau_2025_2040.values * (1 - rate) ** years_elapsed
        for yr, val in zip(SCENARIO_YEARS, projected):
            scenario_rows.append({
                'country':       country,
                'year':          int(yr),
                'scenario':      scenario,
                'co2_projected': round(float(val), 3),
            })

df_scenarios = pd.DataFrame(scenario_rows)
df_scenarios.to_csv('../data/scenario_projections.csv', index=False)
print(f"Saved to data/scenario_projections.csv — {len(df_scenarios)} rows "
      f"({df_scenarios['country'].nunique()} countries × "
      f"{df_scenarios['scenario'].nunique()} scenarios × "
      f"{df_scenarios['year'].nunique()} years)")
display(df_scenarios.head())

<a id='52b-continuity'></a>
### 5.2b Continuity Check — Does BAU Start Where the Data Ends?

A scenario pathway should *continue* the observed series: if its first year sits far from the last observation, the gap is an artifact of the baseline, not an effect of any mitigation. Two thresholds are applied to the first BAU year (2025) against the last observed year (2024): the **aggregate** should be within **±2%** (one year of normal growth is about 1%), and countries outside **±5%** are listed. A few listed countries are expected in a set of ~40 — the point is to see *why* each one is there. The second table scans for the most common cause: a **single-year data anomaly** (a value more than 2× above or below its local median), which this notebook flags but does not alter.

**✏️ Interpretation (2–3 sentences):** Read the aggregate step and the countries listed below. Which of them is explained by a data anomaly, and which by something else?

In [ ]:
# 5.2b Continuity check: first BAU year vs the last observed year
AGGREGATE_TOL_PCT, COUNTRY_TOL_PCT = 2.0, 5.0

last_obs = (
    df_filtered[(df_filtered['year'] == LAST_OBSERVED) & df_filtered['country'].isin(EXPANDED_COUNTRIES)]
    .set_index('country')['co2'].loc[EXPANDED_COUNTRIES]
)
first_bau = pd.Series({c: bau_series(c, SCENARIO_START, SCENARIO_START).iloc[0] for c in EXPANDED_COUNTRIES})
step_pct = (first_bau / last_obs - 1) * 100
aggregate_step = (first_bau.sum() / last_obs.sum() - 1) * 100

status = 'within' if abs(aggregate_step) <= AGGREGATE_TOL_PCT else 'OUTSIDE'
print(f"Aggregate step, {SCENARIO_START} BAU vs {LAST_OBSERVED} observed: {aggregate_step:+.1f}%  ({status} ±{AGGREGATE_TOL_PCT:g}%)")
print(f"Countries within ±{COUNTRY_TOL_PCT:g}%: {(step_pct.abs() <= COUNTRY_TOL_PCT).sum()} of {len(step_pct)} "
      f"(median step {step_pct.median():+.1f}%)")
outside = step_pct[step_pct.abs() > COUNTRY_TOL_PCT].sort_values()
display(outside.round(1).to_frame(f'step vs {LAST_OBSERVED} observed (%)'))

# Single-year anomalies: a value more than 2x above or below the centred 7-year median
anomalies = []
for c in EXPANDED_COUNTRIES:
    s = df_filtered[df_filtered['country'] == c].sort_values('year').set_index('year')['co2']
    ratio = s / s.rolling(7, center=True, min_periods=4).median()
    for yr, r in ratio[(ratio > 2) | (ratio < 0.5)].items():
        anomalies.append({'country': c, 'year': int(yr), 'co2 (Mt)': round(float(s.loc[yr]), 1), 'x local median': round(float(r), 1)})
print('\nSingle-year anomalies (>2x or <0.5x the local median):')
display(pd.DataFrame(anomalies))

**✏️ Interpretation:** The aggregate step is **+0.9%** — within ±2% and about one year of ordinary growth — so BAU continues from the observed 2024 total, where Week 4's export would have started 3.7% above it. Three countries fall outside ±5%. **Kuwait (−13.8%) is a data anomaly:** its 1991 value (493 Mt, ten times its local median — the Gulf War oil fires) sits at the start of the training window and drives its level-smoothing parameter down to 0.145 (a typical country is about 0.85), so its fitted level lags the observed 2024 value; training from 1993 would give +1.0%. **The United Kingdom (−5.4%) and the Netherlands (−5.8%) are not data anomalies** — the scan finds none for them. Both are falling series whose latest year did not keep pace with the trend (the UK's 2024 value rose 1.7% on 2023 against a long decline; the Netherlands' fell only 1.9% against a recent pace of about −8% a year), so the damped trend, continuing the decline from a fitted level, starts below the last observation — an ordinary property of the method, and both stay within 6%. ⚠️ **Worth a second look:** the Netherlands' BAU keeps falling to about 19.5 MtCO₂ by 2040 (−82% from 2025), a steep extrapolation of its recent decline that a damped trend does not flatten quickly, so its scenario figures below should be read with caution.

<a id='53-viz'></a>
### 5.3 Scenario Visualisations

Per-country: one chart overlaying all 3 scenarios (2025–2040) + grey historical reference (1990–2024) + dashed horizontal line at 1990 emissions level. Plotted for the 10 `FEATURED_COUNTRIES` only, in a fixed 5×2 grid (same reasoning as Week 3 §3.8 / Week 4 §4.3 — a 40-panel grid would not be readable).

**Color convention:** BAU = blue | Moderate = orange | Aggressive = green

Also produce a **global aggregate chart**: sum every expanded-set country per scenario per year — unlike the per-country grid, a single summed line per scenario has no per-panel readability limit, so this uses the full ~40-country pool (~92% of actual global CO₂) rather than just the 10 featured countries.

In [ ]:
SCENARIO_COLORS = {'BAU': 'blue', 'Moderate': 'orange', 'Aggressive': 'green'}

# Per-country overlay charts — FEATURED_COUNTRIES (10) only, fixed 5x2 grid.
# Grey = observed history to 2024; the three coloured lines are the scenarios from 2025
# (BAU is the full-data ETS baseline of §5.2; Moderate and Aggressive apply −2% / −5% per year to it).
fig, axes = plt.subplots(5, 2, figsize=(10, 20))
axes_flat = axes.flatten()

for ax, country in zip(axes_flat, FEATURED_COUNTRIES):
    hist = (
        df_filtered[df_filtered['country'] == country]
        .sort_values('year').set_index('year')['co2']
    )
    hist_1990_2024 = hist.loc[1990:2024]
    level_1990 = hist.loc[1990]

    ax.plot(hist_1990_2024.index, hist_1990_2024.values,
            color='grey', linewidth=1, label='Historical (1990–2024)')
    ax.axhline(y=level_1990, color='grey', linestyle='--', linewidth=1, alpha=0.6,
               label='1990 level')

    for scenario, color in SCENARIO_COLORS.items():
        series = (
            df_scenarios[
                (df_scenarios['country'] == country) &
                (df_scenarios['scenario'] == scenario)
            ].set_index('year')['co2_projected']
        )
        ax.plot(series.index, series.values, color=color, linewidth=1.5, label=scenario)

    ax.set_title(country, fontsize=9)
    ax.set_xlabel('Year', fontsize=7)
    ax.set_ylabel('CO₂ (MtCO₂)', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(axis='y', alpha=0.3)

axes_flat[0].legend(fontsize=6, loc='upper left')
plt.suptitle('Scenario Analysis: BAU vs Moderate vs Aggressive Mitigation (history to 2024, scenarios 2025–2040)', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# Global aggregate chart — every expanded-set country summed (not just the 10 featured):
# a single summed line per scenario has no per-panel readability limit, so this uses the
# full pool (~92% of actual global CO2) rather than the curated 10.
hist_global = (
    df_filtered[df_filtered['country'].isin(EXPANDED_COUNTRIES)]
    .groupby('year')['co2'].sum()
    .loc[1990:2024]
)
level_1990_global = hist_global.loc[1990]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hist_global.index, hist_global.values, color='grey', linewidth=1.2,
        label='Historical (1990–2024)')
ax.axhline(y=level_1990_global, color='grey', linestyle='--', linewidth=1, alpha=0.6,
           label='1990 level')

for scenario, color in SCENARIO_COLORS.items():
    series_global = (
        df_scenarios[df_scenarios['scenario'] == scenario]
        .groupby('year')['co2_projected'].sum()
    )
    ax.plot(series_global.index, series_global.values, color=color, linewidth=2, label=scenario)

ax.set_title(f'Global Aggregate ({len(EXPANDED_COUNTRIES)}-Country Expanded-Set Sum): Scenario Comparison', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('CO₂ (MtCO₂)')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

<a id='54-impact'></a>
### 5.4 Impact Summary

Compute cumulative CO₂ per country per scenario (2025–2040 sum). Plot as a grouped bar chart.


In [ ]:
# Cumulative CO₂ 2025–2040 per country per scenario (every expanded-set country — a bar
# chart, unlike the overlay grid above, scales fine with more categories)
df_cumulative = (
    df_scenarios.groupby(['country', 'scenario'])['co2_projected']
    .sum()
    .unstack('scenario')[['BAU', 'Moderate', 'Aggressive']]
    .loc[EXPANDED_COUNTRIES]
)
df_cumulative['Aggressive_pct_reduction'] = (
    (df_cumulative['BAU'] - df_cumulative['Aggressive']) / df_cumulative['BAU'] * 100
)
display(df_cumulative.round(1))

# Grouped bar chart
x = np.arange(len(EXPANDED_COUNTRIES))
width = 0.25

fig, ax = plt.subplots(figsize=(max(12, len(EXPANDED_COUNTRIES) * 0.45), 6))
for i, (scenario, color) in enumerate(SCENARIO_COLORS.items()):
    ax.bar(x + (i - 1) * width, df_cumulative[scenario], width,
           label=scenario, color=color)

ax.set_xticks(x)
ax.set_xticklabels(EXPANDED_COUNTRIES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Cumulative CO₂ 2025–2040 (MtCO₂)')
ax.set_title('Cumulative Emissions 2025–2040 by Scenario')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**✏️ Interpretation (3–5 sentences):** Which countries benefit most from aggressive mitigation in absolute and percentage terms?

**China** shows by far the largest absolute cumulative reduction under Aggressive mitigation — about 69,100 MtCO₂ saved versus BAU over 2025–2040 (a 31.0% cut) — followed by the United States, India, Russia, and Indonesia, the five largest-BAU countries in the expanded set, since a fixed −5%/year compounding rate removes more total tonnage wherever the BAU base is largest. (Japan, fifth when this notebook used Week 4's forecast as BAU, is now seventh: under the full-data baseline Japan's BAU keeps declining — −9.4% over 2025–2040 — while Indonesia's grows by 53.5%, giving Indonesia the larger base.) In percentage terms, **Philippines** (32.3%) and **Indonesia** (32.1%) reduce by a *larger share* than any of the original 10 featured countries, edging out **India** (31.9%, the largest among the original 10), with Bangladesh (31.7%) and Qatar, Algeria and Vietnam (each 31.5%) just behind. **The Netherlands shows the smallest percentage reduction of any country in the expanded set (23.0%), followed by France (27.1%), Germany (28.3%), the United Kingdom (28.8%), and Poland (29.2%)** — all five countries whose BAU *declines* between 2025 and 2040 — a reminder that the mechanism is purely mechanical: a fixed % compounding rate on a *declining* BAU base compounds against smaller and smaller absolute yearly quantities, so the relative gap between BAU and Aggressive narrows faster than for countries with flat or growing BAU trajectories (the Netherlands' very low figure also reflects its steep extrapolated BAU decline, −82% over the period, flagged in §5.2b). This pattern — large emitters and high-growth economies show the biggest absolute *and* percentage gains from aggressive mitigation, while already-declining economies show comparatively modest relative gains — is a direct consequence of applying one uniform rate to different-shaped BAU curves, not evidence that aggressive mitigation matters less for countries like the UK. As emphasized in §5.1, these rankings reflect the illustrative flat-rate assumption applied here, not a measure of each country's actual differentiated mitigation potential or policy ambition.

<a id='55-sensitivity'></a>
### 5.5 Parameter Sensitivity *(Stretch Goal)*

For one country of your choice, show how cumulative 2025–2040 emissions change as the Moderate reduction rate varies from 1% to 4% in 1% steps.

| Reduction Rate | Cumulative CO₂ 2025–2040 (MtCO₂) | % Reduction vs BAU |
|---------------|----------------------------------|-------------------|
| 1% | | |
| 2% | | |
| 3% | | |
| 4% | | |


In [ ]:
# Sensitivity table for one selected country
SENSITIVITY_COUNTRY = 'Brazil'

bau_country = bau_series(SENSITIVITY_COUNTRY, 2025, 2040)
years_elapsed = np.arange(len(bau_country))
bau_cumulative = bau_country.sum()

sensitivity_rows = []
for rate_pct in [1, 2, 3, 4]:
    rate = rate_pct / 100
    projected = bau_country.values * (1 - rate) ** years_elapsed
    cumulative = projected.sum()
    pct_reduction = (bau_cumulative - cumulative) / bau_cumulative * 100
    sensitivity_rows.append({
        'Reduction Rate':                  f'{rate_pct}%',
        'Cumulative CO2 2025-2040 (MtCO2)': round(float(cumulative), 1),
        '% Reduction vs BAU':               round(float(pct_reduction), 2),
    })

df_sensitivity = pd.DataFrame(sensitivity_rows)
print(f"Parameter sensitivity for {SENSITIVITY_COUNTRY} "
      f"(BAU cumulative 2025–2040: {bau_cumulative:,.1f} MtCO₂)")
display(df_sensitivity)


---
> ```
> git add notebook/week5_scenarios.ipynb data/scenario_projections.csv
> git commit -m "Week 5: scenario analysis complete"
> git push
> ```


---
<a id='week6'></a>
# Week 6 — Notebook Finalisation and Optional Streamlit Dashboard

**Learning Objective:** Finalise the notebook to professional standards; optionally build the dashboard.

### 6.1 Notebook Finalisation Checklist

- [ ] Every section has an **opening markdown** (what + why) and a **closing summary** (key findings)
- [ ] **Table of Contents** at the top links to all completed sections
- [ ] All charts: consistent colour schemes, font sizes, and labelling
- [ ] All debugging `print()` statements removed
- [ ] **Kernel → Restart & Run All** — zero errors top to bottom

### 6.2 Streamlit App *(Stretch Goal)*

Open `app.py` (already scaffolded in this repository) and fill in the `# TODO:` sections using Plotly Express charts and your generated data files.

> **Prerequisite:** `data/ghg_features.csv` must exist. The app gracefully handles missing files.

```bash
streamlit run app.py
```

**Minimum interactivity required:**
- `st.selectbox` for country selection
- `st.multiselect` for gas type
- `st.radio` for scenario selection (if Week 5 complete)
- All charts: **Plotly Express** (not Matplotlib)

### 6.3 GitHub Repository Requirements

- [ ] `notebook/week1_eda.ipynb` … `notebook/week5_scenarios.ipynb` — all cells run cleanly
- [ ] `app.py` — runs without errors
- [ ] `data/ghg_features.csv` and (if applicable) `data/scenario_projections.csv`
- [ ] `requirements.txt` — complete
- [ ] `README.md` — team names filled in, instructions verified

---
> ```
> git add .
> git commit -m "Week 6: notebook finalised, Streamlit app added"
> git push
> ```


---
<a id='week7'></a>
# Week 7 — Final Presentation

**Learning Objective:** Deliver a structured 1-hour presentation demonstrating the full analytical workflow.

> *The project report format and formal submission requirements are provided separately.*

### Presentation Outline (60 minutes)

| Slot | Duration | Content |
|------|----------|---------|
| Introduction | 5 min | Project objective, team intro, dataset overview |
| EDA Findings | 10 min | 3 charts + written observations |
| Feature Engineering | 5 min | Key features, rolling mean and lag rationale |
| Model Comparison | 15 min | Live notebook — Baseline → LR → RF → ETS(A,Ad,N), consolidated table |
| Forecasts | 10 min | 3 country forecast plots, summary table, trend interpretation |
| Scenario Analysis | 5 min | Key finding *(if completed)* |
| Streamlit Demo | 5 min | Live app run *(if completed)* |
| Key Takeaways & Q&A | 5 min | Limitations, learnings, next steps |

### Final Submission Checklist

- [ ] Jupyter Notebook: fully documented, all cells run top to bottom without errors
- [ ] Streamlit app *(if built)*: runs without errors, `requirements.txt` complete
- [ ] GitHub repository link **shared before the session**
- [ ] Presentation slides **shared before the session**
- [ ] Project report: submitted in the required report format